In [1]:
import importlib
import models.xvae, models.dec, models.metrics, misc.dataset, misc.helpers
importlib.reload(models.xvae)
importlib.reload(models.dec)
importlib.reload(models.metrics)
importlib.reload(misc.dataset)
importlib.reload(misc.helpers)

from misc.dataset import get_data
from misc.helpers import normalizeRNA
from models.xvae import xvae
from models.dec import XDEC
from models.metrics import calculate_metrics

data = get_data()
x_num = normalizeRNA(data['rnanp'])
x_bin = data['clin']

ae = xvae(s1_input_size=x_num.shape[1], s2_input_size=x_bin.shape[1], ls=16)
ae.build_model()
ae.train(x_num, x_bin)

xdec = XDEC(ae, n_clusters=2)
y_pred, y_proba, centroids, z = xdec.fit(x_num, x_bin, y=data['y'])

acc, ari, nmi = calculate_metrics(data['y'], y_pred)
print('acc={:.3f} ari={:.3f} nmi={:.3f}'.format(acc, ari, nmi))

import os, pandas as pd, torch
os.makedirs('results/manual_run', exist_ok=True)
ae.save_encoder('results/manual_run/encoder_xvae.pt')

emb = pd.DataFrame({'sample_id': data['sample_id']})
for j in range(z.shape[1]): emb['z{}'.format(j)] = z[:, j]
emb['mgs_level'] = data['label_classes'][data['y']]
emb.to_csv('results/manual_run/xdec_latent_embedding.csv', index=False)

# sanity check: confirm the checkpoint is in the new SHAP-compatible format
ckpt_check = torch.load('results/manual_run/encoder_xvae.pt', map_location='cpu')
assert isinstance(ckpt_check, dict) and 's1_input_size' in ckpt_check, (
    "encoder_xvae.pt is still the OLD format (bare state_dict) - "
    "restart the kernel and rerun this cell before using it in the SHAP notebook."
)
print('Encoder checkpoint OK, ready for SHAP:',
     {k: v for k, v in ckpt_check.items() if k != 'state_dict'})

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Brayan Gutierrez\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

proj = PCA(n_components=2, random_state=5192).fit_transform(z)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for c in np.unique(y_pred):
    mask = y_pred == c
    axes[0].scatter(proj[mask, 0], proj[mask, 1], label=f'cluster {c}', alpha=0.7)
axes[0].set_title('X-DEC latent space - predicted clusters')
axes[0].legend()

for c, name in enumerate(data['label_classes']):
    mask = data['y'] == c
    axes[1].scatter(proj[mask, 0], proj[mask, 1], label=name, alpha=0.7)
axes[1].set_title('X-DEC latent space - true mgs_level')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
ct = pd.crosstab(pd.Series(y_pred, name='cluster'),
                 pd.Series(data['label_classes'][data['y']], name='mgs_level'))
print(ct)

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(ct.values, cmap='Blues')
ax.set_xticks(range(len(ct.columns))); ax.set_xticklabels(ct.columns)
ax.set_yticks(range(len(ct.index))); ax.set_yticklabels(ct.index)
ax.set_xlabel('mgs_level'); ax.set_ylabel('cluster')
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        ax.text(j, i, ct.values[i, j], ha='center', va='center')
plt.colorbar(im)
plt.title('X-DEC cluster vs. mgs_level')
plt.show()

In [ ]:
plt.figure(figsize=(5, 4))
plt.hist(y_proba.max(axis=1), bins=20)
plt.xlabel('max soft-assignment probability')
plt.ylabel('# samples')
plt.title('X-DEC cluster assignment confidence')
plt.show()